# =============================================================================
# TAZAMA — ALERT NAVIGATOR & RULE TRIGGER DISCOVERY NOTEBOOK
# Delta Analytics Fellowship — April 2026
# =============================================================================

In [19]:
# -----------------------------------------------------------------------------
# CELL 1: IMPORTS
# -----------------------------------------------------------------------------

In [20]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    TimestampType, DoubleType
)
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
from IPython.display import display, Markdown
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

In [21]:
# -----------------------------------------------------------------------------
# CELL 2: SPARK SESSION
# -----------------------------------------------------------------------------

In [22]:
spark_jars    = os.environ.get("SPARK_JARS", "")
jar_list      = spark_jars.split(",") if spark_jars else []
s3a_endpoint  = os.environ.get("S3A_ENDPOINT", "")
s3a_access    = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret    = os.environ.get("S3A_SECRET_KEY", "")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-RuleTriggerDiscovery")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")


Spark version: 3.4.2


In [23]:
# -----------------------------------------------------------------------------
# CELL 3: LAKEHOUSE PATHS
# All paths confirmed from Gold Layer directory listing.
# WAREHOUSE_ROOT confirmed as /opt/Tazama_Warehouse
# -----------------------------------------------------------------------------

In [24]:
# -----------------------------------------------------------------------------
# CELL 3: LAKEHOUSE PATHS
# Primary data sources for Rule Trigger Discovery:
#   bronze/alerts — full evaluation result with rules, weights, network map
#   gold/alerts   — alert dropdown (fast, clean, flattened)
#   gold/pacs002  — rule query transactions
#   gold/pacs008  — Phase 2 rules (purpose code, remittance info)
#   gold/rules    — band thresholds and rule descriptions
# -----------------------------------------------------------------------------
WAREHOUSE_ROOT      = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")

bronze_alerts_path  = f"{WAREHOUSE_ROOT}/bronze/alerts"
gold_alerts_path    = f"{WAREHOUSE_ROOT}/gold/alerts"
gold_pacs002_path   = f"{WAREHOUSE_ROOT}/gold/pacs002"
gold_pacs008_path   = f"{WAREHOUSE_ROOT}/gold/pacs008"
gold_rules_path     = f"{WAREHOUSE_ROOT}/gold/rule"
gold_evaluation_path = f"{WAREHOUSE_ROOT}/gold/evaluation"
bronze_pacs008_path = f"{WAREHOUSE_ROOT}/bronze/pacs008"
gold_account_holder_path = f"{WAREHOUSE_ROOT}/gold/account_holder"


print("Lakehouse paths configured:")
print(f"  Bronze alerts: {bronze_alerts_path}")
print(f"  Gold alerts:   {gold_alerts_path}")
print(f"  pacs002:       {gold_pacs002_path}")
print(f"  pacs008:       {gold_pacs008_path}")
print(f"  Rules:         {gold_rules_path}")
print(f"  Evaluation:    {gold_evaluation_path}")
print(f"  Bronze_pacs008:{bronze_pacs008_path}")
print(f"  Gold_account_holder:{gold_account_holder_path}")


Lakehouse paths configured:
  Bronze alerts: /opt/Tazama_Warehouse/bronze/alerts
  Gold alerts:   /opt/Tazama_Warehouse/gold/alerts
  pacs002:       /opt/Tazama_Warehouse/gold/pacs002
  pacs008:       /opt/Tazama_Warehouse/gold/pacs008
  Rules:         /opt/Tazama_Warehouse/gold/rule
  Evaluation:    /opt/Tazama_Warehouse/gold/evaluation
  Bronze_pacs008:/opt/Tazama_Warehouse/bronze/pacs008
  Gold_account_holder:/opt/Tazama_Warehouse/gold/account_holder


In [25]:
# -----------------------------------------------------------------------------
# CELL 4: LOAD HUDI TABLES
# pacs002 and pacs008 are the primary tables for Rule Discovery.
# The generic transactions table is NOT used — pacs002/pacs008 have all columns.
# -----------------------------------------------------------------------------

In [26]:
# -----------------------------------------------------------------------------
# CELL 4: LOAD HUDI TABLES
# bronze/alerts — primary source for full evaluation results
# gold/alerts   — alert dropdown only
# gold/pacs002  — all rule queries
# gold/pacs008  — Phase 2 rules
# gold/rules    — band thresholds and rule descriptions
# Note: gold/typologies and gold/transactions are NOT used in this notebook
# -----------------------------------------------------------------------------
try:
    bronze_alerts = spark.read.format("hudi").load(bronze_alerts_path)
    alerts        = spark.read.format("hudi").load(gold_alerts_path)
    pacs002       = spark.read.format("hudi").load(gold_pacs002_path)
    pacs008       = spark.read.format("hudi").load(gold_pacs008_path)
    rules         = spark.read.format("hudi").load(gold_rules_path)
    evaluation = spark.read.format("hudi").load(gold_evaluation_path)
    bronze_pacs008 = spark.read.format("hudi").load(bronze_pacs008_path)
    account_holder = spark.read.format("hudi").load(gold_account_holder_path)

    

    print("All tables loaded successfully.")
    print(f"  bronze_alerts: {bronze_alerts.count():,} records")
    print(f"  alerts:        {alerts.count():,} records")
    print(f"  pacs002:       {pacs002.count():,} records")
    print(f"  pacs008:       {pacs008.count():,} records")
    print(f"  rules:         {rules.count():,} records")
    print(f"  evaluation:   {evaluation.count():,} records")
    print(f"  bronze_pacs008: {bronze_pacs008.count():,} records")
    print(f"  account_holder: {account_holder.count():,} records")

except Exception as e:
    print(f"WARNING: Table load failed — {e}")
    print("Check WAREHOUSE_ROOT and sandbox access before proceeding.")

current_timestamp = datetime.now()
print(f"\n✓ Notebook loaded at: {current_timestamp.strftime('%Y-%m-%d %H:%M:%S UTC')}")


All tables loaded successfully.
  bronze_alerts: 378 records


  alerts:        378 records
  pacs002:       403 records
  pacs008:       402 records
  rules:         510 records
  evaluation:   378 records
  bronze_pacs008: 402 records
  account_holder: 1,186 records

✓ Notebook loaded at: 2026-06-19 05:58:20 UTC


In [27]:
# -----------------------------------------------------------------------------
# CELL 5: STANDARD COLUMN DEFINITIONS
# All columns confirmed from pacs002 and pacs008 Gold Layer schemas.
#
# pacs002 confirmed columns:
#   MsgId       → tx_msg_id
#   EndToEndId  → end_to_end_id
#   DbtrId      → dc_dbtr_id
#   DbtrAcctId  → dc_dbtr_acct_id
#   CdtrId      → dc_cdtr_id
#   CdtrAcctId  → dc_cdtr_acct_id
#   CreDtTm     → dc_cre_dt_tm
#   TxTp        → tx_type
#   TxSts       → tx_status
#   Amt         → tx_amount
#   Ccy         → tx_ccy
#   lat         → None (not in Gold Layer — deferred)
#   long        → None (not in Gold Layer — deferred)
#
# pacs008 confirmed columns:
#   MsgId       → message_id
#   EndToEndId  → end_to_end_id
#   DbtrId      → dc_dbtr_id
#   DbtrAcctId  → dc_dbtr_acct_id
#   CdtrId      → dc_cdtr_id
#   CdtrAcctId  → dc_cdtr_acct_id
#   CreDtTm     → dc_cre_dt_tm
#   TxTp        → tx_type
#   Amt         → instd_amt
#   Ccy         → instd_ccy
# -----------------------------------------------------------------------------

In [28]:
STANDARD_COLS_PACS002 = [
    F.col("tx_msg_id").alias("MsgId"),
    F.col("end_to_end_id").alias("EndToEndId"),
    F.col("dc_dbtr_id").alias("DbtrId"),
    F.col("dc_dbtr_acct_id").alias("DbtrAcctId"),
    F.col("dc_cdtr_id").alias("CdtrId"),
    F.col("dc_cdtr_acct_id").alias("CdtrAcctId"),
    F.col("dc_cre_dt_tm").cast("string").alias("CreDtTm"),  # cast to string
    F.col("tx_type").alias("TxTp"),
    F.col("tx_status").alias("TxSts"),
    F.col("tx_amount").alias("Amt"),
    F.col("tx_ccy").alias("Ccy"),
]
STANDARD_COLS_PACS008 = [
    F.col("message_id").alias("MsgId"),
    F.col("end_to_end_id").alias("EndToEndId"),
    F.col("dc_dbtr_id").alias("DbtrId"),
    F.col("dc_dbtr_acct_id").alias("DbtrAcctId"),
    F.col("dc_cdtr_id").alias("CdtrId"),
    F.col("dc_cdtr_acct_id").alias("CdtrAcctId"),
    F.col("dc_cre_dt_tm").cast("string").alias("CreDtTm"),  # cast to string
    F.col("tx_type").alias("TxTp"),
    F.lit("ACCC").alias("TxSts"),
    F.col("instd_amt").alias("Amt"),
    F.col("instd_ccy").alias("Ccy"),
]


print(f"\n✓ Dashboard generated at: {current_timestamp.strftime('%Y-%m-%d %H:%M:%S UTC')}")


✓ Dashboard generated at: 2026-06-19 05:58:20 UTC


In [29]:
# -----------------------------------------------------------------------------
# CELL 6: ALERT INPUT EXTRACTION
# Extracts inputs from two sources:
#   gold/alerts row — basic alert metadata
#   bronze/alerts alert_data JSON — full evaluation result
# -----------------------------------------------------------------------------

def extract_alert_inputs(eval_row):
    """
    Extract rule query inputs from a single gold/evaluation row.
    All debtor/creditor IDs and transaction amount come directly
    from evaluation table — no pacs002 join needed for IDs.
    """
    return {
        "alertId":      eval_row["evaluation_id"],
        "tenantId":     eval_row["tenant_id"],
        "alertCreDtTm": eval_row["dc_cre_dt_tm"],
        "txMsgId":      eval_row["tx_msg_id"],
        "txOrigE2eId":  eval_row["tx_orgnl_e2e_id"],
        "txAmount":     float(eval_row["dc_instd_amt"]) if eval_row["dc_instd_amt"] else 0.0,
        "txCcy":        str(eval_row["dc_instd_ccy"]) if eval_row["dc_instd_ccy"] else "",
        "txStatus":     eval_row["tx_status"],
        "reportStatus": eval_row["report_status"],
        # Debtor/creditor IDs directly from evaluation — no pacs002 join needed
        "dbtrAcctId":   eval_row["dc_dbtr_acct_id"],
        "cdtrAcctId":   eval_row["dc_cdtr_acct_id"],
        "dbtrId":       eval_row["dc_dbtr_id"],
        "cdtrId":       eval_row["dc_cdtr_id"],
    }


def extract_bronze_alert(evaluation_id, tenant_id):
    """
    Fetch full evaluation result from bronze/alerts.
    Joins via evaluationID field inside alert_data JSON.
    Returns parsed alert_data JSON containing:
      - status (ALRT/NALT)
      - typologyResult with score, thresholds, ruleResults
      - evaluationID
    """
    try:
        rows = bronze_alerts.filter(
            F.col("tenant_id") == tenant_id
        ).select("alert_id", "alert_data").collect()

        for row in rows:
            alert_data_str = row["alert_data"]
            if alert_data_str:
                alert_data = json.loads(alert_data_str)
                if alert_data.get("evaluationID") == evaluation_id:
                    return alert_data
        return None
    except Exception as e:
        print(f"WARNING: Could not fetch bronze alert — {e}")
        return None


def extract_rule_results(alert_data):
    """
    Extract rule results from ALL typologies in alert_data JSON.
    Deduplicates rules — keeps the highest weight per rule.
    Excludes EFRuP rule processor from main results.
    Returns tuple: (rule_results, efrp_ref)
    """
    if not alert_data:
        return [], "none"
    try:
        typology_results = alert_data.get("tadpResult", {}).get("typologyResult", [])
        if not typology_results:
            return [], "none"

        # Collect all rule results across all typologies
        all_rules = {}
        efrp_ref = "none"

        for t in typology_results:
            for r in t.get("ruleResults", []):
                rule_id = r["id"]

                # Extract EFRuP separately instead of just skipping it
                if rule_id == "EFRuP@1.0.0":
                    efrp_ref = r.get("subRuleRef", "none")
                    continue

                # Keep highest weight version if rule appears in multiple typologies
                if rule_id not in all_rules or r.get("wght", 0) > all_rules[rule_id]["weight"]:
                    all_rules[rule_id] = {
                        "rule_id":       rule_id,
                        "weight":        r.get("wght", 0),
                        "sub_rule_ref":  r.get("subRuleRef", ""),
                        "indpdnt_varbl": r.get("indpdntVarbl", 0),
                    }

        return list(all_rules.values()), efrp_ref

    except Exception as e:
        print(f"WARNING: Could not extract rule results — {e}")
        return [], "none"


from IPython.display import HTML

def display_efrp_status(efrp_ref):
    """
    Display EFRuP status with colour coding above the typologies list.
      none                  → green  (no override applied)
      override              → orange (transaction was overridden)
      overridable-block     → orange (block that can be overridden)
      non-overridable-block → red    (hard block, cannot be overridden)
    """
    efrp_colour = {
        "none":                   "green",
        "override":               "orange",
        "overridable-block":      "orange",
        "non-overridable-block":  "red",
    }.get(efrp_ref, "grey")

    display(HTML(
        f"<p><strong>EFRuP Status:</strong> "
        f"<span style='color:{efrp_colour}; font-weight:bold'>{efrp_ref.capitalize()}</span></p>"
    ))


def extract_typology_info(alert_data):
    """
    Extract ALL typology results from parsed alert_data JSON.
    Returns list of dicts — one per typology that was evaluated.
    """
    if not alert_data:
        return []
    try:
        typology_results = alert_data.get("tadpResult", {}).get("typologyResult", [])
        if not typology_results:
            return []
        status = alert_data.get("status", "")
        return [
            {
                "cfg":                   t.get("cfg", ""),
                "score":                 t.get("result", 0),
                "alertThreshold":        t.get("workflow", {}).get("alertThreshold", 0),
                "interdictionThreshold": t.get("workflow", {}).get("interdictionThreshold", 0),
                "status":                status,
            }
            for t in typology_results
        ]
    except Exception as e:
        print(f"WARNING: Could not extract typology info — {e}")
        return []

In [30]:
# -----------------------------------------------------------------------------
# CELL 7: RULE CONFIGURATION HELPERS
# All read from rules variable (loaded from gold/rule path in Cell 4).
# Filter by rule_updated_date descending — replaces as_of_date in new DLH.
# Bands now stored as JSON in bands_json column — not flattened columns.
# -----------------------------------------------------------------------------

def get_maxQueryRange(rule_id, tenant_id):
    """Read max_query_range_ms from rules table."""
    try:
        rule_row = (
            rules
            .filter(F.col("rule_id")   == rule_id)
            .filter(F.col("tenant_id") == tenant_id)
            .orderBy(F.col("rule_updated_date").desc())
            .limit(1)
            .collect()
        )
        if rule_row and rule_row[0]["max_query_range_ms"] is not None:
            return int(rule_row[0]["max_query_range_ms"])
    except Exception as e:
        print(f"WARNING: Could not read max_query_range_ms for {rule_id}: {e}")
    return None



def get_rule_bands(rule_id, tenant_id):
    """
    Read band thresholds from rules table.
    Handles three config structures:
      - bands        : standard numbered bands (.01, .02 etc.)
      - exitConditions: exit/error conditions (.x01, .x04, .err etc.)
      - cases        : expression-based rules like Rule-078 (MP2P, MP2B etc.)
    """
    try:
        rule_row = (
            rules
            .filter(F.col("rule_id")   == rule_id)
            .filter(F.col("tenant_id") == tenant_id)
            .orderBy(F.col("rule_updated_date").desc())
            .limit(1)
            .collect()
        )
        if not rule_row:
            return []

        bands = []

        # Parse normal bands
        bands_json_str = rule_row[0]["bands_json"]
        if bands_json_str:
            for i, b in enumerate(json.loads(bands_json_str)):
                bands.append({
                    "band":         f"Band {i+1}",
                    "reason":       b.get("reason", ""),
                    "sub_rule_ref": b.get("subRuleRef", ""),
                    "lower_limit":  b.get("lowerLimit", None),
                    "upper_limit":  b.get("upperLimit", None),
                })

        # Parse exit conditions
        exit_json_str = rule_row[0]["exit_conditions_json"]
        if exit_json_str:
            for e in json.loads(exit_json_str):
                bands.append({
                    "band":         "Exit condition",
                    "reason":       e.get("reason", ""),
                    "sub_rule_ref": e.get("subRuleRef", ""),
                    "lower_limit":  None,
                    "upper_limit":  None,
                })

        # Parse cases (Rule-078 style — transaction type expressions)
        cases_json_str = rule_row[0]["cases_json"]
        if cases_json_str:
            cases = json.loads(cases_json_str)
            # Add the alternative (fallback) case
            if "alternative" in cases:
                alt = cases["alternative"]
                bands.append({
                    "band":         "Alternative",
                    "reason":       alt.get("reason", ""),
                    "sub_rule_ref": alt.get("subRuleRef", ""),
                    "lower_limit":  None,
                    "upper_limit":  None,
                })
            # Add each expression as a band
            for e in cases.get("expressions", []):
                bands.append({
                    "band":         e.get("value", ""),
                    "reason":       e.get("reason", ""),
                    "sub_rule_ref": e.get("subRuleRef", ""),
                    "lower_limit":  None,
                    "upper_limit":  None,
                })

        return bands

    except Exception as e:
        print(f"WARNING: Could not read bands for {rule_id}: {e}")
    return []


def get_rule_description(rule_id, tenant_id):
    """Read rule_desc from rules table. Falls back to RULE_DESCRIPTIONS dict."""
    try:
        row = (
            rules
            .filter(F.col("rule_id")   == rule_id)
            .filter(F.col("tenant_id") == tenant_id)
            .orderBy(F.col("rule_updated_date").desc())
            .limit(1)
            .collect()
        )
        if row and row[0]["rule_desc"] is not None:
            return row[0]["rule_desc"]
    except:
        pass
    return RULE_DESCRIPTIONS.get(rule_id, "No description available.")

In [31]:
# -----------------------------------------------------------------------------
# CELL 8: TIME BUCKET UTILITIES (reused from TMS dashboard)
# -----------------------------------------------------------------------------

In [32]:
def cast_timestamps_to_string(df):
    """Cast all timestamp columns to string before toPandas()
    to avoid timezone-aware datetime conversion errors."""
    for field in df.schema.fields:
        if isinstance(field.dataType, TimestampType):
            df = df.withColumn(field.name, F.col(field.name).cast("string"))
    return df


In [33]:
# -----------------------------------------------------------------------------
# CELL 9: RULE QUERY FUNCTIONS
# All queries use pacs002 Gold Layer table directly.
# CONFIRMED filter columns: dc_dbtr_acct_id, dc_cdtr_acct_id, tx_status,
# tenant_id, dc_cre_dt_tm.
# CONFIRMED rule ID format: {number}@{version} e.g. 002@1.0.0
# cast_timestamps_to_string() applied to all queries before .toPandas()
# -----------------------------------------------------------------------------

In [34]:
def query_rule_030(inputs):
    """
    Rule-030: Transfer to Unfamiliar Creditor.
    Count of confirmed successful pacs.002 ACCC transactions between
    this specific debtor-creditor pair up to the alert date.
    No time window — full history. No parameters.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_dbtr_acct_id") == inputs["dbtrAcctId"])
        .filter(F.col("dc_cdtr_acct_id") == inputs["cdtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} transaction(s) found"


def query_rule_044(inputs):
    """
    Rule-044: Debtor Lifetime Transaction Count.
    All confirmed successful pacs.002 ACCC transactions for the debtor
    up to and including the alert timestamp.
    Low count = new or inactive debtor account.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_dbtr_acct_id") == inputs["dbtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} transaction(s) found"


def query_rule_045(inputs):
    """
    Rule-045: Creditor Lifetime Transaction Count.
    All confirmed successful pacs.002 ACCC transactions to the creditor
    up to and including the alert timestamp.
    Low count = new or potentially synthetic creditor account.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_cdtr_acct_id") == inputs["cdtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} transaction(s) found"


def query_rule_002(inputs):
    """
    Rule-002: Debtor Transaction Volume in Time Window.
    Confirmed successful pacs.002 ACCC transactions for the debtor
    within max_query_range_ms milliseconds before the alert.
    CONFIRMED: max_query_range_ms = 86,400,000 ms (24 hours).
    """
    max_ms = get_maxQueryRange(inputs["ruleId"], inputs["tenantId"])
    if max_ms is None:
        return pd.DataFrame(), 0, "No time window configuration found"
    window_start = inputs["alertCreDtTm"] - timedelta(milliseconds=max_ms)
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_dbtr_acct_id") == inputs["dbtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    >= F.lit(window_start))
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} transaction(s) found"


def query_rule_016(inputs):
    """
    Rule-016: Transaction Convergence — Creditor.
    Confirmed successful pacs.002 ACCC transactions to the creditor
    within max_query_range_ms milliseconds before the alert.
    CONFIRMED: max_query_range_ms = 86,400,000 ms (24 hours).
    """
    max_ms = get_maxQueryRange(inputs["ruleId"], inputs["tenantId"])
    if max_ms is None:
        return pd.DataFrame(), 0, "No time window configuration found"
    window_start = inputs["alertCreDtTm"] - timedelta(milliseconds=max_ms)
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_cdtr_acct_id") == inputs["cdtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    >= F.lit(window_start))
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} transaction(s) found"


def query_rule_017(inputs):
    """
    Rule-017: Transaction Divergence — Debtor.
    Confirmed successful pacs.002 ACCC transactions from the debtor
    within max_query_range_ms milliseconds before the alert.
    CONFIRMED: max_query_range_ms = 28,800,000 ms (8 hours).
    """
    max_ms = get_maxQueryRange(inputs["ruleId"], inputs["tenantId"])
    if max_ms is None:
        return pd.DataFrame(), 0, "No time window configuration found"
    window_start = inputs["alertCreDtTm"] - timedelta(milliseconds=max_ms)
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_dbtr_acct_id") == inputs["dbtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    >= F.lit(window_start))
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} transaction(s) found"


def query_rule_003(inputs):
    """
    Rule-003: Time Since Most Recent Creditor Activity.
    Most recent confirmed successful pacs.002 ACCC transaction received
    by this creditor account PRIOR to the alerting transaction.
    Independent variable = time gap in milliseconds.
    Exit condition .x01 fires when no prior creditor activity exists.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_cdtr_acct_id") == inputs["cdtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <  F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
        .limit(1)
    ).toPandas()

    if df.empty:
        return df, 0, "No prior creditor activity found (exit condition .x01)"

    last_tx_time = pd.to_datetime(df.iloc[0]["CreDtTm"])
    alert_time   = pd.to_datetime(str(inputs["alertCreDtTm"]))
    gap_ms       = int((alert_time - last_tx_time).total_seconds() * 1000)
    gap_hours    = round(gap_ms / 3600000, 2)
    gap_days     = round(gap_ms / 86400000, 2)

    return df, gap_ms, f"{gap_ms:,} ms ({gap_hours}h / {gap_days} days) since last creditor activity"


def query_rule_076(inputs):
    """
    Rule-076: Time Since Last Transaction — Debtor.
    Two most recent confirmed successful pacs.002 ACCC transactions for
    this debtor account up to and including the alert timestamp.
    Row 0 = alerting transaction. Row 1 = immediately preceding transaction.
    Independent variable = time gap in milliseconds between them.
    Exit condition .x01 fires when only 1 transaction exists (no prior history).
    No time window — full history. No parameters.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_dbtr_acct_id") == inputs["dbtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
        .limit(2)
    ).toPandas()

    # Exit condition .x01 — no prior debtor transaction found
    if len(df) < 2:
        return df, 0, "No prior debtor transaction found (exit condition .x01)"

    # Calculate time gap between alerting and preceding transaction
    current_tx_time = pd.to_datetime(df.iloc[0]["CreDtTm"])
    prior_tx_time   = pd.to_datetime(df.iloc[1]["CreDtTm"])
    gap_ms          = int((current_tx_time - prior_tx_time).total_seconds() * 1000)
    gap_hours       = round(gap_ms / 3600000, 2)
    gap_days        = round(gap_ms / 86400000, 2)

    return df, gap_ms, f"{gap_ms:,} ms ({gap_hours}h / {gap_days} days) since prior debtor transaction"

def query_rule_001(inputs):
    """
    Rule-001: Creditor Account Age.
    Finds the very first confirmed successful pacs.002 ACCC transaction
    ever recorded for this creditor account (MIN dc_cre_dt_tm).
    Independent variable = account age in milliseconds.
    Exit condition .x01 fires when no creditor transaction history exists.
    No time window — full history. No parameters.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_cdtr_acct_id") == inputs["cdtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <= F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").asc())
        .limit(1)
    ).toPandas()

    if df.empty:
        return df, 0, "No creditor transaction history found (exit condition .x01)"

    # IV = account age in milliseconds
    earliest_dt = pd.to_datetime(df.iloc[0]["CreDtTm"])
    alert_dt    = pd.to_datetime(str(inputs["alertCreDtTm"]))
    age_ms      = int((alert_dt - earliest_dt).total_seconds() * 1000)
    age_days    = round(age_ms / 86400000, 2)
    age_hours   = round(age_ms / 3600000, 2)

    return df, age_ms, f"{age_ms:,} ms ({age_hours}h / {age_days} days) — creditor account age"

def query_rule_091(inputs):
    """
    Rule-091: Transaction Amount vs Regulatory Threshold.
    Compares the current transaction amount against the regulatory
    reporting threshold configured in the rules table.
    Band 1: amount below threshold — within regulatory limits
    Band 2: amount above threshold — exceeds regulatory threshold
    Independent variable = transaction amount.
    No database query needed — amount read directly from inputs.
    Threshold read dynamically from rules table per tenant.
    No time window — no parameters.
    """
    tx_amount = float(inputs.get("txAmount", 0) or 0)
    tx_ccy    = inputs.get("txCcy", "")

    # Read threshold dynamically from rules table
    threshold = 10000.0  # fallback default
    try:
        rule_row = (
            rules
            .filter(F.col("rule_id")   == "091@1.0.0")
            .filter(F.col("tenant_id") == inputs["tenantId"])
            .orderBy(F.col("as_of_date").desc())
            .limit(1)
            .collect()
        )
        if rule_row and rule_row[0]["band_02_lower_limit"] is not None:
            threshold = float(rule_row[0]["band_02_lower_limit"])
    except:
        pass

    # Determine band
    if tx_amount >= threshold:
        band_msg = f"⚠️ Exceeds regulatory threshold of {threshold:,.0f}"
    else:
        band_msg = f"✅ Within regulatory limits (threshold: {threshold:,.0f})"

    # Return empty DataFrame — no transaction history to display
    df = pd.DataFrame()

    return df, tx_amount, f"{tx_amount:,.2f} {tx_ccy} — {band_msg}"

def query_rule_078(inputs):
    """
    Rule-078: Transaction Type.
    Single row lookup against pacs008 (not pacs002) returning the
    purpose code (purp_cd) for the alerting transaction.
    Join key: end_to_end_id = txOrigE2eId (not orgnl_end_to_end_id).
    No counting, no aggregation, no time window, no parameters.
    Independent variable = purpose code string (e.g. PEER, BILL, MCHT).
    """
    df = cast_timestamps_to_string(
        pacs008
        .filter(F.col("end_to_end_id") == inputs["txOrigE2eId"])
        .filter(F.col("tenant_id")     == inputs["tenantId"])
        .select(STANDARD_COLS_PACS008 + [F.col("purp_cd").alias("PurpCd")])
        .limit(1)
    ).toPandas()

    if df.empty:
        return df, "—", "No pacs008 record found for this transaction"

    purp_cd = str(df.iloc[0]["PurpCd"]) if pd.notna(df.iloc[0]["PurpCd"]) else "—"
    return df, purp_cd, f"Purpose code: {purp_cd}"


def query_rule_004(inputs):
    """
    Rule-004: Time Since Most Recent Debtor Activity.
    NOTE: Similar to Rule-076 — both measure time gap between
    most recent prior debtor transaction and the alerting transaction.
    Flagged to Justus for clarification on intended difference.
    """
    df = cast_timestamps_to_string(
        pacs002
        .filter(F.col("dc_dbtr_acct_id") == inputs["dbtrAcctId"])
        .filter(F.col("tx_status")       == "ACCC")
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .filter(F.col("dc_cre_dt_tm")    <  F.lit(inputs["alertCreDtTm"]))
        .select(STANDARD_COLS_PACS002)
        .orderBy(F.col("CreDtTm").desc())
        .limit(1)
    ).toPandas()
    if df.empty:
        return df, 0, "No prior debtor activity found (exit condition .x01)"
    last_tx_time = pd.to_datetime(df.iloc[0]["CreDtTm"])
    alert_time   = pd.to_datetime(str(inputs["alertCreDtTm"]))
    gap_ms       = int((alert_time - last_tx_time).total_seconds() * 1000)
    gap_hours    = round(gap_ms / 3600000, 2)
    gap_days     = round(gap_ms / 86400000, 2)
    return df, gap_ms, f"{gap_ms:,} ms ({gap_hours}h / {gap_days} days) since last debtor activity"

def query_rule_028(inputs):
    """
    Rule-028: Age Classification — Debtor.

    Per spec §5.19:
      - Data source: _rawHistory pool (bronze/pacs008) — NOT the gold pacs008.
      - Query inputs:
          * EndToEndId — matches OrgnlEndToEndId from the alerting pacs.002
          * TenantId   — scopes to correct tenant
      - Returned column:
          * BirthDt — extracted from JSON path
            FIToFICstmrCdtTrf.CdtTrfTxInf.Dbtr.Id.PrvtId.DtAndPlcOfBirth.BirthDt
            (must be YYYY-MM-DD)
      - Post-query processing:
          * Compute debtor's age in whole years as at the transaction's CreDtTm
            (FIToFIPmtSts.GrpHdr.CreDtTm) using year-month-day comparison.
      - Visualization (spec):
          * Display BirthDt, calculated age, and CreDtTm — NOT a list of transactions.

    Exit condition .x01 fires when no matching bronze/pacs008 document is found
    or when BirthDt is not present in the JSON.
    """
    # Cell 11 puts the e2e id into "txOrigE2eId"; accept the older name too.
    e2e_id    = inputs.get("txOrigE2eId") or inputs.get("orgnlEndToEndId")
    tenant_id = inputs["tenantId"]

    if not e2e_id:
        return pd.DataFrame(), 0, "No OrgnlEndToEndId on alert (exit condition .x01)"

    # JSON path defined by the spec
    JSON_PATH = "$.FIToFICstmrCdtTrf.CdtTrfTxInf.Dbtr.Id.PrvtId.DtAndPlcOfBirth.BirthDt"

    # The raw payload may live in either `document` or `document_json` on
    # bronze/pacs008 — try the structured column first, fall back to the
    # other. get_json_object accepts JSON-as-string.
    doc_col = "document_json" if "document_json" in bronze_pacs008.columns else "document"

    row_df = (
        bronze_pacs008
        .filter(F.col("end_to_end_id") == e2e_id)
        .filter(F.col("tenant_id")     == tenant_id)
        .select(
            F.col("end_to_end_id").alias("EndToEndId"),
            F.col("message_id").alias("MsgId"),
            F.get_json_object(F.col(doc_col).cast("string"), JSON_PATH).alias("BirthDt"),
        )
        .limit(1)
        .toPandas()
    )

    if row_df.empty:
        return (
            pd.DataFrame(),
            0,
            f"No bronze/pacs008 document found for EndToEndId={e2e_id} (exit condition .x01)"
        )

    birth_dt_raw = row_df.iloc[0]["BirthDt"]
    if birth_dt_raw is None or (isinstance(birth_dt_raw, float) and pd.isna(birth_dt_raw)) or not str(birth_dt_raw).strip():
        return (
            pd.DataFrame(),
            0,
            "BirthDt not present in pacs.008 JSON (exit condition .x01)"
        )

    # Parse BirthDt and compute age in whole years at alert CreDtTm
    try:
        dob      = pd.to_datetime(str(birth_dt_raw)).date()
        cre_dt   = pd.to_datetime(str(inputs["alertCreDtTm"])).date()
        age_years = cre_dt.year - dob.year - (
            (cre_dt.month, cre_dt.day) < (dob.month, dob.day)
        )
    except Exception as e:
        return pd.DataFrame(), 0, f"BirthDt value '{birth_dt_raw}' is a placeholder — real date of birth not available in synthetic data (exit condition .x01)"

    # Build the single-row display DataFrame required by the spec
    display_df = pd.DataFrame([{
        "BirthDt":        dob.isoformat(),
        "CalculatedAge":  age_years,
        "CreDtTm":        str(inputs["alertCreDtTm"]),
        "EndToEndId":     e2e_id,
    }])

    return display_df, age_years, f"Debtor age = {age_years} years (BirthDt {dob.isoformat()}, CreDtTm {inputs['alertCreDtTm']})"

def query_rule_083(inputs):
    """
    Rule-083: Multiple Accounts Associated with Debtor.
    Counts distinct account_id values linked to the debtor's identity
    in the account_holder Gold Layer table.
    High count = one identity controlling many accounts — money mule signal.
    No time window — full history. No parameters.
    """
    df = cast_timestamps_to_string(
        account_holder
        .filter(F.col("counterparty_id") == inputs["dbtrId"])
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .select("counterparty_id", "account_id", "relationship_type", "tenant_id")
        .orderBy("account_id")
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} account(s) linked to debtor identity"

def query_rule_084(inputs):
    """
    Rule-084: Multiple Accounts Associated with Creditor.
    Counts distinct account_id values linked to the creditor's identity
    in the account_holder Gold Layer table.
    High count = one identity controlling many accounts — money mule signal.
    No time window — full history. No parameters.
    Mirror of Rule-083 but for creditor side.
    """
    df = cast_timestamps_to_string(
        account_holder
        .filter(F.col("counterparty_id") == inputs["cdtrId"])
        .filter(F.col("tenant_id")       == inputs["tenantId"])
        .select("counterparty_id", "account_id", "relationship_type", "tenant_id")
        .orderBy("account_id")
    ).toPandas()
    iv = len(df)
    return df, iv, f"{iv} account(s) linked to creditor identity"

# -----------------------------------------------------------------------------
# DISPATCH MAP AND DICTIONARIES
# -----------------------------------------------------------------------------

RULE_QUERIES = {
    "030@1.0.0": query_rule_030,
    "001@1.0.0": query_rule_001,
    "044@1.0.0": query_rule_044,
    "045@1.0.0": query_rule_045,
    "002@1.0.0": query_rule_002,
    "016@1.0.0": query_rule_016,
    "017@1.0.0": query_rule_017,
    "003@1.0.0": query_rule_003,
    "076@1.0.0": query_rule_076,
    "091@1.0.0": query_rule_091,
    "078@1.0.0": query_rule_078,
    "004@1.0.0": query_rule_004,
    "028@1.0.0": query_rule_028,
    "083@1.0.0": query_rule_083,
    "084@1.0.0": query_rule_084, 
}

RULE_DESCRIPTIONS = {
    "030@1.0.0": "Transfer to unfamiliar creditor account - debtor. Counts confirmed successful transactions between this specific debtor-creditor pair up to the alert date.",
    "001@1.0.0": "Creditor account age in milliseconds. Time elapsed from the first ever confirmed successful transaction to this creditor account up to the alert timestamp. A very young account age signals a newly created or synthetic creditor account.",
    "044@1.0.0": "Successful transactions from the debtor, including the new transaction. Counts all confirmed successful outgoing transactions for this debtor account lifetime.",
    "045@1.0.0": "Successful transactions to the creditor, including the new transaction. Counts all confirmed successful incoming transactions to this creditor account lifetime.",
    "002@1.0.0": "Transaction convergence - debtor. Count of confirmed successful debtor transactions within a rolling 24-hour time window.",
    "016@1.0.0": "Transaction convergence - creditor. Count of confirmed successful incoming transactions to the creditor within a rolling 24-hour time window.",
    "017@1.0.0": "Transaction divergence - debtor. Count of confirmed successful outgoing transactions from the debtor within a rolling 8-hour time window.",
    "003@1.0.0": "Time since most recent creditor activity. Most recent confirmed successful incoming transaction to this creditor account before the alert. A long dormancy gap followed by sudden activity signals potential account takeover or synthetic account reactivation.",
    "076@1.0.0": "Time since last transaction - debtor. Time gap in milliseconds between the alerting transaction and the immediately preceding debtor transaction. A very short gap signals rapid automated activity. A very long gap signals dormant account reactivation.",   
    "091@1.0.0": "Transaction amount vs regulatory threshold. Compares the current transaction amount against the configured regulatory reporting threshold. A transaction exceeding the threshold may require mandatory reporting.",
    "078@1.0.0": "Transaction type — purpose code. Identifies the purpose code of the alerting transaction from pacs008. A purpose code inconsistent with the account profile — for example a peer-to-peer transfer from a business account — can be a fraud indicator.",
    "004@1.0.0": "Time since most recent debtor activity. Most recent confirmed successful outgoing transaction from this debtor account before the alert. A long dormancy gap followed by sudden activity signals potential account takeover or dormant account reactivation.",
    "028@1.0.0": "Age classification — debtor. The debtor's age in years derived from the date of birth on the alerting pacs.008 message. Used to flag transactions originating from accounts held by minors or elderly customers, which carry elevated fraud risk.",
    "083@1.0.0": "Count of distinct accounts linked to the debtor's identity in the account_holder table. A high count indicates one identity controlling many accounts — a money mule signal.",
    "084@1.0.0": "Count of distinct accounts linked to the creditor's identity in the account_holder table. A high count indicates one identity controlling many accounts — a money mule signal.",
}

RULE_DISPLAY_LABELS = {
    "030@1.0.0": "Rule-030 — Transfer to Unfamiliar Creditor",
    "001@1.0.0": "Rule-001 — Creditor Account Age",
    "044@1.0.0": "Rule-044 — Debtor Lifetime Transaction Count",
    "045@1.0.0": "Rule-045 — Creditor Lifetime Transaction Count",
    "002@1.0.0": "Rule-002 — Debtor Transaction Convergence",
    "016@1.0.0": "Rule-016 — Creditor Transaction Convergence",
    "017@1.0.0": "Rule-017 — Debtor Transaction Divergence",
    "003@1.0.0": "Rule-003 — Time Since Most Recent Creditor Activity",
    "076@1.0.0": "Rule-076 — Time Since Last Transaction Debtor",
    "091@1.0.0": "Rule-091 — Transaction Amount vs Regulatory Threshold",
    "078@1.0.0": "Rule-078 — Transaction Type",
    "004@1.0.0": "Rule-004 — Time Since Most Recent Debtor Activity",
    "028@1.0.0": "Rule-028 — Age Classification — Debtor",
    "083@1.0.0": "Rule-083 — Multiple Accounts — Debtor",
    "084@1.0.0": "Rule-084 — Multiple Accounts — Creditor",  
}

RULE_SHORT_NAMES = {
    "030@1.0.0": "Rule-030 — Transfer to Unfamiliar Creditor",
    "001@1.0.0": "Rule-001 — Creditor Account Age",
    "044@1.0.0": "Rule-044 — Debtor Lifetime Transaction Count",
    "045@1.0.0": "Rule-045 — Creditor Lifetime Transaction Count",
    "002@1.0.0": "Rule-002 — Debtor Transaction Convergence",
    "016@1.0.0": "Rule-016 — Creditor Transaction Convergence",
    "017@1.0.0": "Rule-017 — Debtor Transaction Divergence",
    "003@1.0.0": "Rule-003 — Time Since Most Recent Creditor Activity",
    "076@1.0.0": "Rule-076 — Time Since Last Transaction Debtor",
    "091@1.0.0": "Rule-091 — Transaction Amount vs Regulatory Threshold",
    "078@1.0.0": "Rule-078 — Transaction Type",
    "004@1.0.0": "Rule-004 — Time Since Most Recent Debtor Activity",
    "028@1.0.0": "Rule-028 — Age Classification — Debtor",
    "083@1.0.0": "Rule-083 — Multiple Accounts — Debtor",
    "084@1.0.0": "Rule-084 — Multiple Accounts — Creditor",
}

In [35]:
# -----------------------------------------------------------------------------
# CELL 10: VISUALIZATION FUNCTION
# Renders Rule Trigger Discovery output for a selected rule.
# Shows: rule header, description, independent variable, band thresholds,
#        transaction table.
# -----------------------------------------------------------------------------

def display_rule_discovery(rule_id, df, inputs, iv_value=None, iv_label=None):
    """
    Display Rule Trigger Discovery visualization for a single rule.
    rule_id is the clean rule ID e.g. 030@1.0.0
    iv_value — numeric or string independent variable value
    iv_label — human readable IV display text
    """
    # Drop lat and long — not available in Gold Layer
    display_cols = [c for c in df.columns if c not in ["lat", "long"]]
    df_display   = df[display_cols]

    # Default IV to row count if not provided
    if iv_value is None:
        iv_value = len(df)
    if iv_label is None:
        iv_label = f"{iv_value} transaction(s) found"

    # Row count for layout sizing — always numeric regardless of IV type
    n_rows = len(df_display)

    # --- RULE HEADER ---
    rule_number = rule_id.split("@")[0]
    rule_name   = RULE_SHORT_NAMES.get(rule_id, rule_id)
    try:
        row_r = (
            rules
            .filter(F.col("rule_id")   == rule_id)
            .filter(F.col("tenant_id") == inputs["tenantId"])
            .orderBy(F.col("as_of_date").desc())
            .limit(1)
            .collect()
        )
        max_ms = int(row_r[0]["max_query_range_ms"]) if row_r and row_r[0]["max_query_range_ms"] else None
    except:
        max_ms = None

    if max_ms:
        hours       = int(max_ms / 3600000)
        rule_header = f"{rule_name} — {hours}h window"
    else:
        rule_header = f"{rule_name} — Full history"

    display(Markdown(f"## {rule_header}"))
    display(Markdown(f"*{RULE_DESCRIPTIONS.get(rule_id, 'No description available.')}*"))
    display(Markdown(f"**Independent variable: {iv_label}**"))

    # --- BAND THRESHOLDS ---
    bands = get_rule_bands(rule_id, inputs["tenantId"])
    if bands:
        display(Markdown("**Rule band thresholds:**"))
        bands_df = pd.DataFrame(bands)[["band", "reason", "lower_limit", "upper_limit"]]
        bands_df.columns = ["Band", "Reason", "Lower limit", "Upper limit"]
        bands_df["Lower limit"] = bands_df["Lower limit"].apply(
            lambda x: "—" if pd.isna(x) else str(int(x)) if x == int(x) else str(x)
        )
        bands_df["Upper limit"] = bands_df["Upper limit"].apply(
            lambda x: "—" if pd.isna(x) else str(int(x)) if x == int(x) else str(x)
        )
        pd.set_option('display.max_colwidth', None)
        bands_df = bands_df.set_index("Band")
        bands_df.index.name = None
        display(bands_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]))

    # --- NO TRANSACTIONS ---
    if df_display.empty:
        display(Markdown("**No transactions found for this rule and alert.**"))
        return

    # --- PLOTLY TRANSACTION TABLE ---
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(df_display.columns),
            fill_color="#2E75B6",
            font=dict(color="white", size=11),
            align="left",
            height=30
        ),
        cells=dict(
            values=[df_display[c] for c in df_display.columns],
            fill_color=[["#EBF3FB" if i % 2 == 0 else "white" for i in range(n_rows)]],
            font=dict(size=11),
            align="left",
            height=25
        )
    )])

    alert_label = "Alert-" + str(inputs.get('alertId', ''))

    fig.update_layout(
        title=dict(
            text=f"Rule-{rule_number} — {iv_label} | {alert_label} | Tenant: {inputs.get('tenantId', '')}",
            font=dict(size=13)
        ),
        margin=dict(t=50, b=20, l=0, r=0),
        height=max(250, 60 + (n_rows * 28))
    )
    fig.show()

In [36]:
# -----------------------------------------------------------------------------
# CELL 11: ALERT NAVIGATOR + RULE TRIGGER DISCOVERY WIDGET
# Layout:
#   1. Alert dropdown — gold/evaluation table
#   2. Alert metadata — from gold/evaluation
#   3. Typology section — from bronze/alerts via evaluationID join
#   4. Rules triggered — from bronze/alerts via evaluationID join
#   5. Rule dropdown — filtered to POC rules that fired
#   6. Run button
#   7. Rule Trigger Discovery — band thresholds + transaction table
#
# Data sources:
#   gold/evaluation — primary source: dropdown, metadata, debtor/creditor IDs
#   bronze/alerts   — full evaluation result via evaluationID join
#   gold/pacs002    — transaction data for rule queries
#   gold/rule       — band thresholds
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# LOAD REFERENCE DATA
# -----------------------------------------------------------------------------
# Load evaluation table — primary data source
try:
    eval_df = cast_timestamps_to_string(
        evaluation
        .select("evaluation_id", "tenant_id", "report_status",
                "is_alert", "event_ts", "tx_msg_id", "tx_orgnl_e2e_id",
                "tx_type", "tx_status", "dc_instd_amt", "dc_instd_ccy",
                "dc_dbtr_acct_id", "dc_cdtr_acct_id",
                "dc_dbtr_id", "dc_cdtr_id", "dc_cre_dt_tm",
                "typology_count")
        .orderBy(F.col("event_ts").desc())
    ).toPandas()
    print(f"Evaluations loaded: {len(eval_df)} rows")
except Exception as e:
    print(f"WARNING: Could not load evaluation — {e}")
    eval_df = pd.DataFrame()

# Build evaluation maps with friendly labels
eval_id_list = (
    [
        f"{row['tenant_id']} | {row['report_status']} | {str(row['event_ts'])[:19]} | {row['dc_instd_ccy']} {float(row['dc_instd_amt']):,.2f} | {str(row['evaluation_id'])[:8]}..."
        for _, row in eval_df.iterrows()
    ]
    if not eval_df.empty else []
)
eval_row_map = {
    f"{row['tenant_id']} | {row['report_status']} | {str(row['event_ts'])[:19]} | {row['dc_instd_ccy']} {float(row['dc_instd_amt']):,.2f} | {str(row['evaluation_id'])[:8]}...": row
    for _, row in eval_df.iterrows()
}
# -----------------------------------------------------------------------------
# WIDGETS
# -----------------------------------------------------------------------------

alert_filter = widgets.Text(
    placeholder="Type to filter alerts (tenant, status, amount)...",
    description="Filter:",
    layout=widgets.Layout(width="500px")
)

alert_dropdown = widgets.Dropdown(
    options=["-- Select an alert --"] + eval_id_list,
    description="Alert:",
    layout=widgets.Layout(width="600px")
)

def on_filter_changed(change):
    """Filter alert dropdown based on text input."""
    filter_text = change["new"].strip().lower()
    if filter_text == "":
        alert_dropdown.options = ["-- Select an alert --"] + eval_id_list
    else:
        filtered = [a for a in eval_id_list if filter_text in a.lower()]
        alert_dropdown.options = ["-- Select an alert --"] + filtered

alert_filter.observe(on_filter_changed, names="value")

rule_dropdown = widgets.Dropdown(
    options=["-- Select a rule --"],
    description="Rule:",
    layout=widgets.Layout(width="400px")
)

run_button = widgets.Button(
    description="Run Rule Discovery",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)

meta_output  = widgets.Output()
typo_output  = widgets.Output()
rules_output = widgets.Output()
disc_output  = widgets.Output()


# -----------------------------------------------------------------------------
# EVENT HANDLERS
# -----------------------------------------------------------------------------

def on_alert_selected(change):
    """When alert selected — display metadata, typology, rules sections."""
    alert_id = change["new"]

    meta_output.clear_output()
    typo_output.clear_output()
    rules_output.clear_output()
    disc_output.clear_output()
    rule_dropdown.options = ["-- Select a rule --"]

    if alert_id == "-- Select an alert --":
        return

    row = eval_row_map.get(str(alert_id))
    if row is None:
        return

    tenant_id     = row["tenant_id"]
    evaluation_id = str(row["evaluation_id"])

    # Fetch full evaluation result from bronze/alerts via evaluationID
    alert_data   = extract_bronze_alert(evaluation_id, tenant_id)
    typo_info    = extract_typology_info(alert_data)
    rule_results, efrp_ref = extract_rule_results(alert_data)

   

    # --- ALERT METADATA ---
    with meta_output:
        display(Markdown("### Alert metadata"))
        meta_data = {
            "Evaluation ID":    str(row["evaluation_id"]),
            "Transaction ID":   str(row["tx_msg_id"]),
            "Timestamp":        str(row["event_ts"])[:19],
            "Transaction type": str(row["tx_type"]),
            "TX Status":        str(row["tx_status"]),
            "Amount":           f"{row['dc_instd_ccy']} {float(row['dc_instd_amt']):,.2f}" if pd.notna(row["dc_instd_amt"]) and row["dc_instd_amt"] else "—",
            "Report status":    str(row["report_status"]),
            "Typologies":       str(row["typology_count"]),
            "Tenant":           str(row["tenant_id"]),
        }
        display(pd.DataFrame([meta_data]))

    # --- TYPOLOGY SECTION ---
    with typo_output:
        
        display_efrp_status(efrp_ref)
        display(Markdown("### Typologies evaluated"))
        if typo_info:
            typo_rows = []
            for t in typo_info:
                typo_rows.append({
                    "Typology config":        t["cfg"],
                    "Typology Score":         t["score"],
                    "Alert threshold":        t["alertThreshold"],
                    "Interdiction threshold": t["interdictionThreshold"],
                    "Breached":               "✅ Yes" if t["score"] >= t["alertThreshold"] and t["alertThreshold"] > 0 else "❌ No",
                })
            typo_df = pd.DataFrame(typo_rows).sort_values("Typology Score", ascending=False)
            display(typo_df.set_index("Typology config"))
            display(Markdown(f"*Overall status: **{typo_info[0]['status']}***"))
        else:
            display(Markdown("*Typology details not available for this alert.*"))

   # --- RULES TRIGGERED ---
    with rules_output:
        display(Markdown("### Rules triggered"))
        if rule_results:
            rules_data = []
            matched_poc_rules = []

            for r in rule_results:
                rule_id   = r["rule_id"]
                weight    = r["weight"]
                sub_ref   = r["sub_rule_ref"]
                rule_name = RULE_SHORT_NAMES.get(rule_id, f"Rule-{rule_id.split('@')[0]}")

                sub_ref_reason = ""
                try:
                    bands = get_rule_bands(rule_id, tenant_id)
                    matched_band = next(
                        (b for b in bands if b["sub_rule_ref"] == sub_ref), None
                    )
                    if matched_band:
                        sub_ref_reason = matched_band["reason"]
                except:
                    pass
                
                
                if rule_id in RULE_QUERIES:
                    try:
                        inputs = extract_alert_inputs(row)
                        inputs["ruleId"] = rule_id
                        _, iv_value, _ = RULE_QUERIES[rule_id](inputs)
                        iv = iv_value
                    except:
                        iv = r["indpdnt_varbl"]
                else:
                    iv = r["indpdnt_varbl"]

                rules_data.append({
                    "Rule":                 rule_name,
                    "Weight":               weight,
                    "Independent variable": iv,
                    "Sub-rule ref":         sub_ref,
                    "Reason":               sub_ref_reason,
                })

                if rule_id in RULE_QUERIES:
                    matched_poc_rules.append(rule_id)

            rules_df = pd.DataFrame(rules_data).set_index("Rule")
            rules_df = rules_df.sort_values("Weight", ascending=False)
            
            # Displays the entire Reason line instead of cutting off
            with pd.option_context("display.max_colwidth", None):    
                display(rules_df)

            # Filter dropdown — only rules with weight > 0 and implemented
            weighted_poc_rules = [
                r["rule_id"] for r in rule_results
                if r["weight"] > 0 and r["rule_id"] in RULE_QUERIES
            ]

            if weighted_poc_rules:
                rule_dropdown.options = (
                    ["-- Select a rule --"] +
                    [RULE_DISPLAY_LABELS[r] for r in weighted_poc_rules]
                )
            else:
                rule_dropdown.options = (
                    ["-- Select a rule --"] +
                    list(RULE_DISPLAY_LABELS.values())
                )

        else:
            display(Markdown("*Rule results not available.*"))
            rule_dropdown.options = (
                ["-- Select a rule --"] +
                list(RULE_DISPLAY_LABELS.values())
            )
            
def on_run_clicked(b):
    """Run selected rule query and display Rule Trigger Discovery."""
    disc_output.clear_output()

    selected_row = eval_row_map.get(alert_dropdown.value)
    alert_id     = selected_row["evaluation_id"] if selected_row is not None else ""
    rule_display = rule_dropdown.value

    with disc_output:
        if not alert_id or alert_dropdown.value == "-- Select an alert --":
            display(Markdown("**Please select an alert first.**"))
            return
        if rule_display == "-- Select a rule --":
            display(Markdown("**Please select a rule to investigate.**"))
            return

        # Resolve clean rule ID
        rule_id_clean = [
            k for k, v in RULE_DISPLAY_LABELS.items()
            if v == rule_display
        ]
        if not rule_id_clean:
            display(Markdown("**ERROR: Could not resolve rule ID.**"))
            return
        rule_id_clean = rule_id_clean[0]

        rule_num = rule_id_clean.split("@")[0]
        display(Markdown(f"Running **Rule-{rule_num}** for alert **Alert-{alert_id}**..."))

        # Fetch evaluation row — has all debtor/creditor IDs directly
        try:
           eval_row = evaluation.filter(
            F.col("evaluation_id") == str(alert_id)
           ).limit(1).collect()[0]
           inputs = extract_alert_inputs(eval_row)
           inputs["ruleId"] = rule_id_clean

        except Exception as e:
            display(Markdown(f"**ERROR extracting alert inputs:** {e}"))
            return

        # Run rule query — no pacs002 join needed for IDs
        try:
            result = RULE_QUERIES[rule_id_clean](inputs)
            if isinstance(result, tuple):
                df, iv_value, iv_label = result
            else:
                df, iv_value, iv_label = result, len(result), None
        except Exception as e:
            display(Markdown(f"**ERROR running rule query:** {e}"))
            return

        # Display visualization
        display_rule_discovery(rule_id_clean, df, inputs, iv_value, iv_label)


# -----------------------------------------------------------------------------
# WIRE UP EVENTS AND RENDER
# -----------------------------------------------------------------------------
alert_dropdown.observe(on_alert_selected, names="value")
run_button.on_click(on_run_clicked)

display(Markdown("# Tazama — Alert Navigator"))
display(Markdown("*Rule Trigger Discovery — select an alert and investigate the rules that triggered it.*"))
display(widgets.VBox([alert_filter, alert_dropdown]))
display(meta_output)
display(typo_output)
display(rules_output)
display(widgets.HBox([rule_dropdown, run_button]))
display(disc_output)

Evaluations loaded: 378 rows


# Tazama — Alert Navigator

*Rule Trigger Discovery — select an alert and investigate the rules that triggered it.*

Output()

Output()

Output()

Output()